# 🌐 Phase 1 — Multilingual Translator: Dataset Build + Hybrid Model Training

**Architecture:** XLM-RoBERTa encoder (first 6 layers frozen) + Custom 6-layer Transformer decoder built from scratch.

> **This notebook is fully self-contained** — no external dataset input is needed.
> All Python source files are written to `/kaggle/working/` by the cells below.

**Languages covered (24):**
- 🇮🇳 Indian: Hindi, Bengali, Tamil, Telugu, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Urdu
- 🇪🇺 European: French, German, Spanish, Italian, Portuguese, Dutch, Polish, Romanian, Swedish, Czech
- 🌏 CJK: Chinese (Simplified), Japanese, Korean
- 🌐 English (pivot)

**Corpus sources:** `Helsinki-NLP/opus-100`, `opus_books`, `Helsinki-NLP/tatoeba_mt`

**Hardware:** 2× NVIDIA T4 (16 GB each) — enable in Kaggle: *Settings → Accelerator → GPU T4 x2*

---
## Workflow
1. Install dependencies
2. Set HF secrets
3. GPU check
4. Write source scripts to disk  ← **fixes the "No such file" error**
5. Build & upload dataset
6. Train the hybrid model
7. Inspect metrics & checkpoints

## Step 1 — Install Dependencies

In [ ]:
import subprocess, sys

def pip_install(packages, quiet=True):
    q = ["--quiet"] if quiet else []
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade"] + q + packages
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️  pip error:\n{result.stderr[-2000:]}")
    else:
        print(f"✅ {' '.join(packages[:3])}{'...' if len(packages)>3 else ''}")

# Core ML
pip_install(["torch>=2.5.0", "torchvision>=0.20.0", "torchaudio>=2.5.0"])

# HuggingFace ecosystem
pip_install(["transformers>=4.45.0", "datasets>=3.0.0", "tokenizers>=0.20.0",
             "huggingface-hub>=0.25.0", "accelerate>=1.0.0", "evaluate>=0.4.3"])

# Data + metrics
pip_install(["pandas>=2.2.1", "pyarrow>=15.0.2", "sacrebleu>=2.4.0",
             "rouge-score>=0.1.2", "jiwer>=3.0.4"])

# Utilities
pip_install(["langdetect==1.0.9", "tqdm>=4.66.2", "scikit-learn>=1.4.0",
             "matplotlib>=3.8.3", "seaborn>=0.13.2", "pynvml>=11.5.0",
             "fastparquet>=2024.2.0"])

print("\n🚀 All packages installed successfully!")

## Step 2 — Configure Secrets & Environment

In [ ]:
import os

# ── Read from Kaggle Secrets ────────────────────────────────────────────────
# Add secrets in Kaggle: Notebook → Add-ons → Secrets
#   Name: HF_TOKEN    Value: hf_xxxxxxxxxxxx
#   Name: HF_USERNAME Value: your_hf_username
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    HF_TOKEN    = _s.get_secret("HF_TOKEN")
    HF_USERNAME = _s.get_secret("HF_USERNAME")
    print(f"✅ Secrets loaded from Kaggle vault — HF_USERNAME={HF_USERNAME}")
except Exception as e:
    print(f"⚠️  Kaggle secrets unavailable ({e})")
    # ── Fallback: paste manually ────────────────────────────────────────────
    HF_TOKEN    = ""   # 👈 paste your HF token here if secrets don't work
    HF_USERNAME = ""   # 👈 paste your HF username here

if not HF_TOKEN:
    print("❌  HF_TOKEN is empty — dataset/model push to Hub will be skipped.")
if not HF_USERNAME:
    print("❌  HF_USERNAME is empty — Hub repo names will be incomplete.")

# ── Set environment variables for child processes ───────────────────────────
os.environ["HF_TOKEN"]       = HF_TOKEN
os.environ["HF_USERNAME"]    = HF_USERNAME
os.environ["DATA_DIR"]       = "/kaggle/working/multilingual_dataset"
os.environ["CHECKPOINT_DIR"] = "/kaggle/working/checkpoints"
os.environ["OUTPUT_DIR"]     = "/kaggle/working/multilingual_dataset"

print(f"DATA_DIR       = {os.environ['DATA_DIR']}")
print(f"CHECKPOINT_DIR = {os.environ['CHECKPOINT_DIR']}")

## Step 3 — GPU & System Check

In [ ]:
import torch, platform

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Device  : {'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️ (enable GPU in Settings!)'}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB  ({p.multi_processor_count} SMs)")
else:
    print("\n⚠️  No GPU found — go to Settings → Accelerator → GPU T4 x2 and re-run!")

# Disk space
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk /kaggle/working: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

## Step 4 — Write Source Scripts to Disk

> **This is the fix for the 'No such file or directory' error.**
> Both Python scripts are embedded directly in this notebook cell
> and written to `/kaggle/working/` before being executed.

In [ ]:
# ── Write dataset_builder.py to Kaggle working dir ────────────────────────────
import pathlib, textwrap

_SRC = """\"\"\"
Phase 1 — Dataset Builder
=========================
Downloads multilingual parallel corpora from HuggingFace Datasets Hub,
cleans and unifies them into a single Parquet dataset, creates train/val/test
splits, writes dataset statistics, and optionally pushes to HuggingFace Hub.

Usage (Kaggle / CLI):
    python dataset_builder.py

Environment variables:
    HF_TOKEN       — HuggingFace write token (required for hub push)
    HF_USERNAME    — HuggingFace username  (required for hub push)
    OUTPUT_DIR     — Output directory      (default: /kaggle/working/multilingual_dataset)
\"\"\"

import os
import json
import logging
import hashlib
import re
import sys
import time
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

# ── HuggingFace ────────────────────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from huggingface_hub import HfApi, login

# ── Language detection ─────────────────────────────────────────────────────────
try:
    from langdetect import detect, LangDetectException
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("dataset_builder")

# ═══════════════════════════════════════════════════════════════════════════════
# Constants
# ═══════════════════════════════════════════════════════════════════════════════

OUTPUT_DIR   = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working/multilingual_dataset"))
HF_TOKEN     = os.environ.get("HF_TOKEN", "")
HF_USERNAME  = os.environ.get("HF_USERNAME", "")
HF_REPO_ID   = f"{HF_USERNAME}/multilingual-translation-data" if HF_USERNAME else ""

SHARD_SIZE   = 100_000   # rows per Parquet shard
MIN_CHARS    = 10
MAX_CHARS    = 512
TRAIN_RATIO  = 0.90
VAL_RATIO    = 0.05
# TEST = remainder

# ── Language coverage ─────────────────────────────────────────────────────────
INDIAN_LANGS   = ["hi", "bn", "ta", "te", "mr", "gu", "kn", "ml", "pa", "ur"]
EUROPEAN_LANGS = ["fr", "de", "es", "it", "pt", "nl", "pl", "ro", "sv", "cs"]
CJK_LANGS      = ["zh", "ja", "ko"]
PIVOT_LANG     = "en"
ALL_LANGS      = [PIVOT_LANG] + INDIAN_LANGS + EUROPEAN_LANGS + CJK_LANGS

# ── Dataset recipes ────────────────────────────────────────────────────────────
# Each recipe: (hf_dataset_name, config_or_None, src_field_path, tgt_field_path,
#               src_lang_key, tgt_lang_key, domain_tag)
# lang_pair strings follow the pattern expected by each dataset.

OPUS100_PAIRS = [
    # Indian
    ("en-hi", "hi"), ("en-bn", "bn"), ("en-ta", "ta"), ("en-te", "te"),
    ("en-mr", "mr"), ("en-gu", "gu"), ("en-kn", "kn"), ("en-ml", "ml"),
    ("en-pa", "pa"), ("en-ur", "ur"),
    # European
    ("en-fr", "fr"), ("en-de", "de"), ("en-es", "es"), ("en-it", "it"),
    ("en-pt", "pt"), ("en-nl", "nl"), ("en-pl", "pl"), ("en-ro", "ro"),
    ("en-sv", "sv"), ("en-cs", "cs"),
    # CJK
    ("en-zh", "zh"), ("en-ja", "ja"), ("en-ko", "ko"),
]

OPUS_BOOKS_PAIRS = [
    ("en-fr", "fr"), ("en-de", "de"), ("en-es", "es"), ("en-it", "it"),
    ("en-pt", "pt"), ("en-nl", "nl"), ("en-ro", "ro"), ("en-cs", "cs"),
    ("en-ru", None),  # skip ru — not in target langs but keep recipe modular
]

TATOEBA_PAIRS = [
    # Indian subset that tatoeba carries
    ("eng-hin", "hi"), ("eng-ben", "bn"), ("eng-tam", "ta"), ("eng-tel", "te"),
    ("eng-mar", "mr"), ("eng-urd", "ur"),
    # CJK
    ("eng-cmn", "zh"), ("eng-jpn", "ja"), ("eng-kor", "ko"),
    # European
    ("eng-fra", "fr"), ("eng-deu", "de"), ("eng-spa", "es"), ("eng-ita", "it"),
    ("eng-por", "pt"), ("eng-nld", "nl"), ("eng-pol", "pl"),
    ("eng-ron", "ro"), ("eng-swe", "sv"), ("eng-ces", "cs"),
]

# ═══════════════════════════════════════════════════════════════════════════════
# Helper utilities
# ═══════════════════════════════════════════════════════════════════════════════

def _text_hash(src: str, tgt: str) -> str:
    return hashlib.md5(f"{src}|||{tgt}".encode("utf-8")).hexdigest()


def _is_valid_text(text: str, min_c: int = MIN_CHARS, max_c: int = MAX_CHARS) -> bool:
    if not isinstance(text, str):
        return False
    t = text.strip()
    return min_c <= len(t) <= max_c


def _detect_lang(text: str, expected_lang: str) -> bool:
    \"\"\"Return True if detected language matches expected (or detection unavailable).\"\"\"
    if not LANGDETECT_AVAILABLE:
        return True
    # Only validate well-resourced languages to avoid false negatives on scripts
    VALIDATABLE = {"en", "fr", "de", "es", "it", "pt", "nl", "pl", "ro", "sv",
                   "cs", "hi", "bn", "ta", "te", "mr", "gu", "ml", "ko", "zh",
                   "ja", "ur"}
    if expected_lang not in VALIDATABLE:
        return True
    try:
        detected = detect(text)
        # langdetect returns "zh-cn" / "zh-tw" — normalise
        if detected.startswith("zh"):
            detected = "zh"
        return detected == expected_lang
    except LangDetectException:
        return True


def clean_text(text: str) -> str:
    \"\"\"Light normalisation: collapse whitespace, strip leading/trailing spaces.\"\"\"
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\\s+", " ", text)
    text = text.strip()
    return text


# ═══════════════════════════════════════════════════════════════════════════════
# Dataset loaders
# ═══════════════════════════════════════════════════════════════════════════════

def _build_row(src: str, tgt: str, src_lang: str, tgt_lang: str, domain: str,
               seen: set) -> Optional[Dict]:
    src = clean_text(src)
    tgt = clean_text(tgt)
    if not _is_valid_text(src) or not _is_valid_text(tgt):
        return None
    h = _text_hash(src, tgt)
    if h in seen:
        return None
    seen.add(h)
    return {
        "source_text":  src,
        "target_text":  tgt,
        "source_lang":  src_lang,
        "target_lang":  tgt_lang,
        "domain":       domain,
        "pair":         f"{src_lang}-{tgt_lang}",
    }


def load_opus100(seen: set) -> List[Dict]:
    rows = []
    for lang_pair, tgt_lang in tqdm(OPUS100_PAIRS, desc="opus-100"):
        try:
            ds = load_dataset(
                "Helsinki-NLP/opus-100",
                lang_pair,
                split="train",
                trust_remote_code=True,
            )
            for ex in ds:
                tr = ex.get("translation", {})
                src_text = tr.get("en", "")
                tgt_text = tr.get(tgt_lang, "")
                row = _build_row(src_text, tgt_text, "en", tgt_lang, "opus100", seen)
                if row:
                    rows.append(row)
        except Exception as exc:
            log.warning(f"opus-100 [{lang_pair}] failed: {exc}")
    log.info(f"opus-100 → {len(rows):,} rows")
    return rows


def load_opus_books(seen: set) -> List[Dict]:
    rows = []
    target_tgt_langs = {"fr", "de", "es", "it", "pt", "nl", "ro", "cs"}
    for lang_pair, tgt_lang in tqdm(OPUS_BOOKS_PAIRS, desc="opus_books"):
        if tgt_lang not in target_tgt_langs:
            continue
        try:
            ds = load_dataset(
                "opus_books",
                lang_pair,
                split="train",
                trust_remote_code=True,
            )
            for ex in ds:
                tr = ex.get("translation", {})
                # opus_books keys may be "en"/"fr" etc.
                src_text = tr.get("en", "")
                tgt_text = tr.get(tgt_lang, "")
                row = _build_row(src_text, tgt_text, "en", tgt_lang, "books", seen)
                if row:
                    rows.append(row)
        except Exception as exc:
            log.warning(f"opus_books [{lang_pair}] failed: {exc}")
    log.info(f"opus_books → {len(rows):,} rows")
    return rows


def load_tatoeba(seen: set) -> List[Dict]:
    rows = []
    for lang_pair, tgt_lang in tqdm(TATOEBA_PAIRS, desc="tatoeba_mt"):
        try:
            ds = load_dataset(
                "Helsinki-NLP/tatoeba_mt",
                lang_pair,
                split="test",
                trust_remote_code=True,
            )
            for ex in ds:
                src_text = ex.get("sourceString", "") or ex.get("src", "")
                tgt_text = ex.get("targetString", "") or ex.get("trg", "")
                row = _build_row(src_text, tgt_text, "en", tgt_lang, "tatoeba", seen)
                if row:
                    rows.append(row)
        except Exception as exc:
            log.warning(f"tatoeba_mt [{lang_pair}] failed: {exc}")
    log.info(f"tatoeba_mt → {len(rows):,} rows")
    return rows


# ═══════════════════════════════════════════════════════════════════════════════
# Language-detection validation pass
# ═══════════════════════════════════════════════════════════════════════════════

def validate_language_detection(df: pd.DataFrame, sample_frac: float = 0.05) -> pd.DataFrame:
    \"\"\"
    Run langdetect on a random sample of rows.  Rows that FAIL detection on
    the sample are NOT removed — only logged — to avoid over-filtering low-
    resource languages where langdetect is unreliable.
    \"\"\"
    if not LANGDETECT_AVAILABLE:
        log.info("langdetect not available — skipping validation.")
        return df

    sample = df.sample(frac=sample_frac, random_state=42)
    fail_counts: Dict[str, int] = {}

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="lang-detect validate"):
        for field, lang_col in [("source_text", "source_lang"), ("target_text", "target_lang")]:
            lang = row[lang_col]
            if not _detect_lang(row[field], lang):
                fail_counts[lang] = fail_counts.get(lang, 0) + 1

    if fail_counts:
        log.warning(f"Lang-detect mismatches in sample (not removed): {fail_counts}")
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# Parquet sharded writer
# ═══════════════════════════════════════════════════════════════════════════════

def save_split_parquet(df: pd.DataFrame, split_name: str, out_dir: Path) -> List[Path]:
    split_dir = out_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(df, preserve_index=False)
    n_shards = max(1, (len(df) + SHARD_SIZE - 1) // SHARD_SIZE)
    shard_paths = []
    for shard_idx in range(n_shards):
        start = shard_idx * SHARD_SIZE
        end   = min(start + SHARD_SIZE, len(df))
        shard = table.slice(start, end - start)
        shard_path = split_dir / f"shard_{shard_idx:05d}.parquet"
        pq.write_table(shard, shard_path, compression="snappy")
        shard_paths.append(shard_path)
        log.info(f"  Wrote {shard_path.name}  ({end-start:,} rows)")
    return shard_paths


# ═══════════════════════════════════════════════════════════════════════════════
# Statistics
# ═══════════════════════════════════════════════════════════════════════════════

def compute_statistics(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> Dict:
    total = len(train) + len(val) + len(test)
    stats = {
        "total_rows":        total,
        "train_rows":        len(train),
        "val_rows":          len(val),
        "test_rows":         len(test),
        "language_pairs":    {},
        "domain_distribution": {},
        "source_len_stats":  {},
        "target_len_stats":  {},
    }

    full = pd.concat([train, val, test], ignore_index=True)

    # Per language-pair counts
    for pair, grp in full.groupby("pair"):
        stats["language_pairs"][pair] = int(len(grp))

    # Domain distribution
    for dom, grp in full.groupby("domain"):
        stats["domain_distribution"][dom] = int(len(grp))

    # Length statistics
    for col, key in [("source_text", "source_len_stats"), ("target_text", "target_len_stats")]:
        lengths = full[col].str.len()
        stats[key] = {
            "mean":   float(lengths.mean()),
            "median": float(lengths.median()),
            "std":    float(lengths.std()),
            "min":    int(lengths.min()),
            "max":    int(lengths.max()),
            "p25":    float(lengths.quantile(0.25)),
            "p75":    float(lengths.quantile(0.75)),
            "p95":    float(lengths.quantile(0.95)),
        }

    return stats


# ═══════════════════════════════════════════════════════════════════════════════
# HuggingFace Hub push
# ═══════════════════════════════════════════════════════════════════════════════

def push_to_hub(out_dir: Path, repo_id: str, token: str) -> None:
    if not token or not repo_id:
        log.warning("HF_TOKEN / HF_USERNAME not set — skipping hub push.")
        return

    log.info(f"Logging in to HuggingFace Hub as {HF_USERNAME} …")
    login(token=token)
    api = HfApi()

    # Create repo if it doesn't exist
    try:
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
        log.info(f"Hub repo: {repo_id}")
    except Exception as exc:
        log.error(f"Could not create repo {repo_id}: {exc}")
        return

    # Upload all Parquet shards + stats JSON
    for path in sorted(out_dir.rglob("*")):
        if path.is_file() and path.suffix in {".parquet", ".json"}:
            rel = path.relative_to(out_dir)
            log.info(f"  Uploading {rel} …")
            try:
                api.upload_file(
                    path_or_fileobj=str(path),
                    path_in_repo=str(rel).replace("\\\\", "/"),
                    repo_id=repo_id,
                    repo_type="dataset",
                    token=token,
                )
            except Exception as exc:
                log.warning(f"  Upload failed for {rel}: {exc}")

    log.info("Hub push complete.")


# ═══════════════════════════════════════════════════════════════════════════════
# Main
# ═══════════════════════════════════════════════════════════════════════════════

def main() -> None:
    t0 = time.time()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log.info(f"Output directory: {OUTPUT_DIR}")

    # ── 1. Collect data ───────────────────────────────────────────────────────
    seen: set = set()
    all_rows: List[Dict] = []

    all_rows.extend(load_opus100(seen))
    all_rows.extend(load_opus_books(seen))
    all_rows.extend(load_tatoeba(seen))

    if not all_rows:
        log.error("No rows collected — aborting.")
        sys.exit(1)

    log.info(f"Total collected rows (before cleaning): {len(all_rows):,}")

    # ── 2. Build DataFrame & clean ────────────────────────────────────────────
    df = pd.DataFrame(all_rows)

    # Drop rows with any null in critical columns
    df.dropna(subset=["source_text", "target_text", "source_lang", "target_lang"], inplace=True)

    # Length filter (already done per row but re-apply on DataFrame level)
    src_len = df["source_text"].str.len()
    tgt_len = df["target_text"].str.len()
    df = df[(src_len >= MIN_CHARS) & (src_len <= MAX_CHARS) &
            (tgt_len >= MIN_CHARS) & (tgt_len <= MAX_CHARS)].copy()

    # Dedup by hash of (source_text, target_text)
    df["_hash"] = df["source_text"] + "|||" + df["target_text"]
    df.drop_duplicates(subset="_hash", inplace=True)
    df.drop(columns="_hash", inplace=True)
    df.reset_index(drop=True, inplace=True)

    log.info(f"Rows after cleaning: {len(df):,}")

    # ── 3. Language-detection validation (sample) ─────────────────────────────
    df = validate_language_detection(df, sample_frac=0.02)

    # ── 4. Shuffle & split ────────────────────────────────────────────────────
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    n = len(df)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    train_df = df.iloc[:n_train].copy()
    val_df   = df.iloc[n_train : n_train + n_val].copy()
    test_df  = df.iloc[n_train + n_val :].copy()

    log.info(f"Split sizes — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

    # ── 5. Save Parquet shards ────────────────────────────────────────────────
    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        log.info(f"Writing {split_name} split …")
        save_split_parquet(split_df, split_name, OUTPUT_DIR)

    # ── 6. Dataset statistics ─────────────────────────────────────────────────
    stats = compute_statistics(train_df, val_df, test_df)
    stats["elapsed_seconds"] = round(time.time() - t0, 2)
    stats_path = OUTPUT_DIR / "dataset_statistics.json"
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    log.info(f"Statistics saved → {stats_path}")
    log.info(json.dumps(stats, indent=2))

    # ── 7. Push to HuggingFace Hub ────────────────────────────────────────────
    push_to_hub(OUTPUT_DIR, HF_REPO_ID, HF_TOKEN)

    elapsed = time.time() - t0
    log.info(f"Dataset builder finished in {elapsed:.1f}s")


if __name__ == "__main__":
    main()
"""

dest = pathlib.Path("/kaggle/working/dataset_builder.py")
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(_SRC, encoding="utf-8")
print(f"✅ Written {len(_SRC)} chars → {dest}")


In [ ]:
# ── Write train_hybrid_model.py to Kaggle working dir ────────────────────────────
import pathlib, textwrap

_SRC = """\"\"\"
Phase 1 — Train Hybrid Model: Custom XLM-RoBERTa Encoder + 6-Layer Scratch Decoder
====================================================================================
Model name: custom-hybrid-v1

Architecture
------------
  Encoder : FacebookAI/xlm-roberta-base  (first 6 layers frozen, last 6 fine-tuned)
  Decoder : Custom 6-layer Transformer built from scratch in PyTorch
              d_model=768  nhead=12  dim_feedforward=3072  dropout=0.1
              Language embedding (50 languages, learned)
              Sinusoidal positional encoding
              Output projection → vocab_size

Training
--------
  Mixed precision (fp16)        Gradient checkpointing
  AdamW + cosine LR schedule    Label-smoothing cross-entropy (0.1)
  Gradient accumulation 4 steps Batch size 32 / GPU
  Checkpoint every 1000 steps   Hub push every 5000 steps
  Early stopping (patience=3)   BLEU evaluation on val set

Usage:
    python train_hybrid_model.py

Environment variables:
    HF_TOKEN       — HuggingFace write token
    HF_USERNAME    — HuggingFace username
    DATA_DIR       — Parquet dataset dir (default: /kaggle/working/multilingual_dataset)
    CHECKPOINT_DIR — Where to save checkpoints (default: /kaggle/working/checkpoints)
\"\"\"

from __future__ import annotations

import json
import logging
import math
import os
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset as TorchDataset

import pandas as pd
import pyarrow.parquet as pq
import sacrebleu
from tqdm import tqdm

from transformers import (
    XLMRobertaTokenizerFast,
    XLMRobertaModel,
    get_cosine_schedule_with_warmup,
)
from huggingface_hub import HfApi, login

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("train_hybrid")

# ═══════════════════════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════════════════════

class Config:
    # Paths
    DATA_DIR        = Path(os.environ.get("DATA_DIR",       "/kaggle/working/multilingual_dataset"))
    CHECKPOINT_DIR  = Path(os.environ.get("CHECKPOINT_DIR", "/kaggle/working/checkpoints"))
    HF_TOKEN        = os.environ.get("HF_TOKEN",    "")
    HF_USERNAME     = os.environ.get("HF_USERNAME", "")
    HF_REPO_ID      = f"{HF_USERNAME}/custom-hybrid-v1" if os.environ.get("HF_USERNAME") else ""

    # Model
    ENCODER_MODEL   = "FacebookAI/xlm-roberta-base"
    D_MODEL         = 768
    NHEAD           = 12
    NUM_DEC_LAYERS  = 6
    DIM_FEEDFORWARD = 3072
    DROPOUT         = 0.1
    MAX_SEQ_LEN     = 128   # tokens (keep GPU memory in check on 2×T4)
    NUM_LANGUAGES   = 50    # language embedding slots

    # Training
    BATCH_SIZE          = 32
    GRAD_ACCUM_STEPS    = 4
    LR                  = 3e-4
    ENCODER_LR          = 5e-5   # lower LR for fine-tuned encoder layers
    WARMUP_STEPS        = 1000
    MAX_STEPS           = 100_000
    LABEL_SMOOTHING     = 0.1
    EVAL_EVERY          = 1000
    SAVE_EVERY          = 1000
    HUB_PUSH_EVERY      = 5000
    EARLY_STOP_PATIENCE = 3
    NUM_WORKERS         = 2
    SEED                = 42

    # Encoder layer freezing
    FREEZE_ENCODER_LAYERS = 6   # freeze first N layers (0-indexed)

    LOG_FILE = Path(os.environ.get("CHECKPOINT_DIR", "/kaggle/working/checkpoints")) / "training_log.jsonl"


CFG = Config()


# ── Language-code → index mapping ─────────────────────────────────────────────
LANG_CODES = [
    "en", "hi", "bn", "ta", "te", "mr", "gu", "kn", "ml", "pa", "ur",
    "fr", "de", "es", "it", "pt", "nl", "pl", "ro", "sv", "cs",
    "zh", "ja", "ko",
    # padding slots (allow room for more languages)
    *[f"xx{i}" for i in range(26)],
]
LANG2IDX: Dict[str, int] = {lang: idx for idx, lang in enumerate(LANG_CODES)}


def lang_to_idx(lang: str) -> int:
    return LANG2IDX.get(lang, len(LANG_CODES) - 1)


# ═══════════════════════════════════════════════════════════════════════════════
# Positional Encoding (sinusoidal)
# ═══════════════════════════════════════════════════════════════════════════════

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)   # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ═══════════════════════════════════════════════════════════════════════════════
# Multilingual Decoder (6 layers, built from scratch)
# ═══════════════════════════════════════════════════════════════════════════════

class MultilingualDecoder(nn.Module):
    \"\"\"
    6-layer Transformer decoder that cross-attends to XLM-RoBERTa encoder
    hidden states.  A learned language embedding is summed to every decoder
    input token so the model knows which target language to generate.
    \"\"\"

    def __init__(
        self,
        vocab_size: int,
        d_model: int              = CFG.D_MODEL,
        nhead: int                = CFG.NHEAD,
        num_layers: int           = CFG.NUM_DEC_LAYERS,
        dim_feedforward: int      = CFG.DIM_FEEDFORWARD,
        dropout: float            = CFG.DROPOUT,
        num_languages: int        = CFG.NUM_LANGUAGES,
        max_seq_len: int          = 1024,
    ):
        super().__init__()
        self.d_model = d_model

        # Token + language + position embeddings
        self.token_embedding    = nn.Embedding(vocab_size, d_model, padding_idx=1)
        self.language_embedding = nn.Embedding(num_languages, d_model)
        self.pos_encoding       = SinusoidalPositionalEncoding(d_model, max_seq_len, dropout)

        # 6× TransformerDecoderLayer
        decoder_layer = nn.TransformerDecoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_feedforward,
            dropout         = dropout,
            activation      = "gelu",
            batch_first     = True,
            norm_first      = True,   # Pre-LN for training stability
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers     = num_layers,
            norm           = nn.LayerNorm(d_model),
        )

        # Output projection
        self.output_projection = nn.Linear(d_model, vocab_size, bias=False)

        # Tie weights between token embedding and output projection
        self.output_projection.weight = self.token_embedding.weight

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        nn.init.normal_(self.language_embedding.weight, std=0.02)
        nn.init.zeros_(self.output_projection.bias if self.output_projection.bias is not None else
                       torch.zeros(1))   # no-op guard

    @staticmethod
    def _make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
        \"\"\"Upper-triangular mask for auto-regressive decoding.\"\"\"
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return mask

    def forward(
        self,
        tgt_ids:          torch.Tensor,     # (B, T)
        encoder_out:      torch.Tensor,     # (B, S, d_model)
        tgt_lang_ids:     torch.Tensor,     # (B,)
        encoder_pad_mask: Optional[torch.Tensor] = None,  # (B, S) True = padding
        tgt_key_padding_mask: Optional[torch.Tensor] = None,  # (B, T)
    ) -> torch.Tensor:
        B, T = tgt_ids.shape
        device = tgt_ids.device

        # Build target embeddings
        tok_emb  = self.token_embedding(tgt_ids)             # (B, T, d)
        lang_emb = self.language_embedding(tgt_lang_ids)     # (B, d)
        lang_emb = lang_emb.unsqueeze(1).expand(-1, T, -1)  # (B, T, d)

        x = self.pos_encoding(tok_emb + lang_emb)           # (B, T, d)

        # Causal mask
        causal_mask = self._make_causal_mask(T, device)      # (T, T)

        # Transformer decoder
        out = self.transformer_decoder(
            tgt                  = x,
            memory               = encoder_out,
            tgt_mask             = causal_mask,
            tgt_key_padding_mask = tgt_key_padding_mask,
            memory_key_padding_mask = encoder_pad_mask,
        )   # (B, T, d)

        logits = self.output_projection(out)                 # (B, T, vocab)
        return logits


# ═══════════════════════════════════════════════════════════════════════════════
# Full Hybrid Model
# ═══════════════════════════════════════════════════════════════════════════════

class CustomHybridTranslator(nn.Module):
    \"\"\"
    XLM-RoBERTa encoder (first 6 layers frozen) + Custom 6-layer decoder.
    \"\"\"

    def __init__(self, tokenizer: XLMRobertaTokenizerFast):
        super().__init__()
        self.tokenizer = tokenizer
        vocab_size = len(tokenizer)

        # ── Encoder ───────────────────────────────────────────────────────────
        log.info(f"Loading encoder: {CFG.ENCODER_MODEL}")
        self.encoder: XLMRobertaModel = XLMRobertaModel.from_pretrained(
            CFG.ENCODER_MODEL,
            add_pooling_layer=False,
        )

        # Enable gradient checkpointing on encoder to save VRAM
        self.encoder.gradient_checkpointing_enable()

        # Freeze first N encoder layers
        self._freeze_encoder_layers(CFG.FREEZE_ENCODER_LAYERS)

        # ── Decoder ───────────────────────────────────────────────────────────
        log.info(f"Building decoder: d_model={CFG.D_MODEL} layers={CFG.NUM_DEC_LAYERS}")
        self.decoder = MultilingualDecoder(vocab_size=vocab_size)

    def _freeze_encoder_layers(self, n: int) -> None:
        # Always freeze embeddings
        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False
        # Freeze first n transformer layers
        for i, layer in enumerate(self.encoder.encoder.layer):
            if i < n:
                for param in layer.parameters():
                    param.requires_grad = False
        frozen = sum(1 for p in self.encoder.parameters() if not p.requires_grad)
        total  = sum(1 for p in self.encoder.parameters())
        log.info(f"Encoder: {frozen}/{total} parameter groups frozen (first {n} layers + embeddings)")

    def encode(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state   # (B, S, d_model)

    def forward(
        self,
        src_input_ids:   torch.Tensor,   # (B, S)
        src_attn_mask:   torch.Tensor,   # (B, S)
        tgt_input_ids:   torch.Tensor,   # (B, T)
        tgt_lang_ids:    torch.Tensor,   # (B,)
        tgt_attn_mask:   Optional[torch.Tensor] = None,  # (B, T)
    ) -> torch.Tensor:
        encoder_out = self.encode(src_input_ids, src_attn_mask)
        # Encoder pad mask: True where padding
        encoder_pad_mask = (src_attn_mask == 0)
        # Decoder pad mask: True where padding
        tgt_pad_mask = (tgt_attn_mask == 0) if tgt_attn_mask is not None else None

        logits = self.decoder(
            tgt_ids              = tgt_input_ids,
            encoder_out          = encoder_out,
            tgt_lang_ids         = tgt_lang_ids,
            encoder_pad_mask     = encoder_pad_mask,
            tgt_key_padding_mask = tgt_pad_mask,
        )
        return logits   # (B, T, vocab)

    @torch.no_grad()
    def generate(
        self,
        src_input_ids: torch.Tensor,
        src_attn_mask: torch.Tensor,
        tgt_lang_id:   int,
        max_new_tokens: int = 128,
        bos_token_id:   int = 0,
        eos_token_id:   int = 2,
    ) -> List[List[int]]:
        \"\"\"Greedy decoding for BLEU evaluation.\"\"\"
        device = src_input_ids.device
        B = src_input_ids.size(0)
        encoder_out = self.encode(src_input_ids, src_attn_mask)
        encoder_pad_mask = (src_attn_mask == 0)

        lang_ids = torch.full((B,), tgt_lang_id, dtype=torch.long, device=device)
        tgt = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        sequences = [[] for _ in range(B)]

        for _ in range(max_new_tokens):
            logits = self.decoder(
                tgt_ids          = tgt,
                encoder_out      = encoder_out,
                tgt_lang_ids     = lang_ids,
                encoder_pad_mask = encoder_pad_mask,
            )
            next_token = logits[:, -1, :].argmax(dim=-1)   # (B,)
            for i in range(B):
                if not finished[i]:
                    sequences[i].append(next_token[i].item())
                    if next_token[i].item() == eos_token_id:
                        finished[i] = True
            tgt = torch.cat([tgt, next_token.unsqueeze(1)], dim=1)
            if finished.all():
                break

        return sequences


# ═══════════════════════════════════════════════════════════════════════════════
# Dataset
# ═══════════════════════════════════════════════════════════════════════════════

class TranslationDataset(TorchDataset):
    \"\"\"
    Reads sharded Parquet files from a split directory.
    Tokenises on-the-fly (CPU) to keep GPU busy.
    \"\"\"

    def __init__(
        self,
        split_dir:   Path,
        tokenizer:   XLMRobertaTokenizerFast,
        max_seq_len: int = CFG.MAX_SEQ_LEN,
    ):
        self.tokenizer   = tokenizer
        self.max_seq_len = max_seq_len
        self.rows: List[Dict] = []

        shards = sorted(split_dir.glob("*.parquet"))
        if not shards:
            raise FileNotFoundError(f"No parquet shards in {split_dir}")

        log.info(f"Loading {len(shards)} shards from {split_dir} …")
        for shard in tqdm(shards, desc=f"load {split_dir.name}"):
            tbl = pq.read_table(shard, columns=["source_text", "target_text", "source_lang", "target_lang"])
            df  = tbl.to_pandas()
            for _, row in df.iterrows():
                self.rows.append({
                    "src":      row["source_text"],
                    "tgt":      row["target_text"],
                    "src_lang": row["source_lang"],
                    "tgt_lang": row["target_lang"],
                })

        log.info(f"  → {len(self.rows):,} examples loaded")

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.rows[idx]

        src_enc = self.tokenizer(
            item["src"],
            max_length=self.max_seq_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        tgt_enc = self.tokenizer(
            item["tgt"],
            max_length=self.max_seq_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        src_ids  = src_enc["input_ids"].squeeze(0)        # (S,)
        src_mask = src_enc["attention_mask"].squeeze(0)   # (S,)
        tgt_ids  = tgt_enc["input_ids"].squeeze(0)        # (T,)
        tgt_mask = tgt_enc["attention_mask"].squeeze(0)   # (T,)

        # Decoder input: shift right (prepend BOS, drop last token)
        dec_input = torch.cat([tgt_ids.new_tensor([self.tokenizer.bos_token_id]),
                               tgt_ids[:-1]])
        dec_mask  = torch.cat([tgt_mask.new_tensor([1]), tgt_mask[:-1]])

        return {
            "src_input_ids":  src_ids,
            "src_attn_mask":  src_mask,
            "tgt_input_ids":  dec_input,       # decoder input
            "tgt_attn_mask":  dec_mask,
            "labels":         tgt_ids,         # ground truth (shifted)
            "tgt_lang_id":    torch.tensor(lang_to_idx(item["tgt_lang"]), dtype=torch.long),
        }


# ═══════════════════════════════════════════════════════════════════════════════
# Loss
# ═══════════════════════════════════════════════════════════════════════════════

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing: float = 0.1, ignore_index: int = 1):
        super().__init__()
        self.smoothing     = smoothing
        self.ignore_index  = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: (B, T, V)   targets: (B, T)
        B, T, V = logits.shape
        logits  = logits.reshape(-1, V)
        targets = targets.reshape(-1)

        log_probs = F.log_softmax(logits, dim=-1)
        nll_loss  = -log_probs.gather(dim=-1, index=targets.clamp(min=0).unsqueeze(-1)).squeeze(-1)
        smooth_loss = -log_probs.mean(dim=-1)

        loss = (1.0 - self.smoothing) * nll_loss + self.smoothing * smooth_loss

        # Mask padding
        mask = targets != self.ignore_index
        loss = loss[mask]
        return loss.mean()


# ═══════════════════════════════════════════════════════════════════════════════
# BLEU evaluation helper
# ═══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_bleu(
    model:     CustomHybridTranslator,
    loader:    DataLoader,
    tokenizer: XLMRobertaTokenizerFast,
    device:    torch.device,
    max_batches: int = 50,
) -> float:
    model.eval()
    hypotheses: List[str] = []
    references: List[str] = []

    for batch_idx, batch in enumerate(loader):
        if batch_idx >= max_batches:
            break
        src_ids  = batch["src_input_ids"].to(device)
        src_mask = batch["src_attn_mask"].to(device)
        labels   = batch["labels"]                         # (B, T) on CPU
        tgt_lang = batch["tgt_lang_id"][0].item()         # use first item (batch may be mixed)

        seqs = model.generate(
            src_input_ids  = src_ids,
            src_attn_mask  = src_mask,
            tgt_lang_id    = tgt_lang,
            max_new_tokens = 64,
            bos_token_id   = tokenizer.bos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

        for i, seq in enumerate(seqs):
            hyp = tokenizer.decode(seq, skip_special_tokens=True)
            ref_ids = labels[i].tolist()
            ref = tokenizer.decode(ref_ids, skip_special_tokens=True)
            hypotheses.append(hyp)
            references.append(ref)

    if not hypotheses:
        return 0.0

    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score


# ═══════════════════════════════════════════════════════════════════════════════
# Checkpoint helpers
# ═══════════════════════════════════════════════════════════════════════════════

def save_checkpoint(
    model:     CustomHybridTranslator,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler:    GradScaler,
    step:      int,
    val_loss:  float,
    ckpt_dir:  Path,
) -> Path:
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / f"checkpoint_step_{step:07d}.pt"
    # If model is wrapped in DataParallel
    state_dict = (model.module if hasattr(model, "module") else model).state_dict()
    torch.save(
        {
            "step":          step,
            "val_loss":      val_loss,
            "model":         state_dict,
            "optimizer":     optimizer.state_dict(),
            "scheduler":     scheduler.state_dict(),
            "scaler":        scaler.state_dict(),
        },
        ckpt_path,
    )
    log.info(f"Checkpoint saved → {ckpt_path}")
    return ckpt_path


def push_checkpoint_to_hub(ckpt_path: Path, repo_id: str, token: str) -> None:
    if not token or not repo_id:
        log.warning("HF_TOKEN / HF_USERNAME not set — skipping hub push.")
        return
    login(token=token)
    api = HfApi()
    try:
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
        api.upload_file(
            path_or_fileobj = str(ckpt_path),
            path_in_repo    = ckpt_path.name,
            repo_id         = repo_id,
            repo_type       = "model",
            token           = token,
        )
        log.info(f"Checkpoint pushed to hub: {repo_id}/{ckpt_path.name}")
    except Exception as exc:
        log.warning(f"Hub push failed: {exc}")


# ═══════════════════════════════════════════════════════════════════════════════
# Training log
# ═══════════════════════════════════════════════════════════════════════════════

def log_metrics(metrics: Dict, log_file: Path) -> None:
    log_file.parent.mkdir(parents=True, exist_ok=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(metrics) + "\\n")


# ═══════════════════════════════════════════════════════════════════════════════
# Main training loop
# ═══════════════════════════════════════════════════════════════════════════════

def build_optimizer_and_scheduler(model: nn.Module, total_steps: int):
    # Separate param groups: encoder (low LR) vs decoder (higher LR)
    base = model.module if hasattr(model, "module") else model
    encoder_params = [p for p in base.encoder.parameters() if p.requires_grad]
    decoder_params = list(base.decoder.parameters())

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": CFG.ENCODER_LR},
            {"params": decoder_params, "lr": CFG.LR},
        ],
        betas=(0.9, 0.98),
        eps=1e-8,
        weight_decay=0.01,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps    = CFG.WARMUP_STEPS,
        num_training_steps  = total_steps,
    )
    return optimizer, scheduler


def train() -> None:
    torch.manual_seed(CFG.SEED)
    np.random.seed(CFG.SEED)
    CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    # ── Device setup ──────────────────────────────────────────────────────────
    if torch.cuda.is_available():
        n_gpus = torch.cuda.device_count()
        device = torch.device("cuda")
        log.info(f"Using {n_gpus} GPU(s): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}")
    else:
        device = torch.device("cpu")
        n_gpus = 0
        log.warning("No GPU found — training on CPU (will be slow).")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    log.info(f"Loading tokenizer: {CFG.ENCODER_MODEL}")
    tokenizer: XLMRobertaTokenizerFast = XLMRobertaTokenizerFast.from_pretrained(CFG.ENCODER_MODEL)

    # ── Datasets ──────────────────────────────────────────────────────────────
    train_dataset = TranslationDataset(CFG.DATA_DIR / "train", tokenizer)
    val_dataset   = TranslationDataset(CFG.DATA_DIR / "val",   tokenizer)

    effective_batch = CFG.BATCH_SIZE * max(1, n_gpus) * CFG.GRAD_ACCUM_STEPS
    log.info(f"Effective batch size: {effective_batch}")

    train_loader = DataLoader(
        train_dataset,
        batch_size  = CFG.BATCH_SIZE * max(1, n_gpus),
        shuffle     = True,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = device.type == "cuda",
        drop_last   = True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size  = CFG.BATCH_SIZE * max(1, n_gpus),
        shuffle     = False,
        num_workers = CFG.NUM_WORKERS,
        pin_memory  = device.type == "cuda",
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    model = CustomHybridTranslator(tokenizer).to(device)
    if n_gpus > 1:
        model = nn.DataParallel(model)
        log.info(f"Wrapped model in DataParallel across {n_gpus} GPUs.")

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"Model params — total: {total_params/1e6:.1f}M  trainable: {trainable_params/1e6:.1f}M")

    # ── Loss / optim / scheduler / scaler ────────────────────────────────────
    criterion = LabelSmoothingCrossEntropy(
        smoothing    = CFG.LABEL_SMOOTHING,
        ignore_index = tokenizer.pad_token_id,
    )
    optimizer, scheduler = build_optimizer_and_scheduler(model, CFG.MAX_STEPS)
    scaler = GradScaler(enabled=(device.type == "cuda"))

    # ── Training state ────────────────────────────────────────────────────────
    global_step       = 0
    best_val_loss     = float("inf")
    no_improve_count  = 0
    t_start           = time.time()

    log.info("Starting training …")

    while global_step < CFG.MAX_STEPS:
        model.train()

        for batch in train_loader:
            if global_step >= CFG.MAX_STEPS:
                break

            src_ids  = batch["src_input_ids"].to(device)
            src_mask = batch["src_attn_mask"].to(device)
            tgt_ids  = batch["tgt_input_ids"].to(device)
            tgt_mask = batch["tgt_attn_mask"].to(device)
            labels   = batch["labels"].to(device)
            lang_ids = batch["tgt_lang_id"].to(device)

            with autocast(enabled=(device.type == "cuda")):
                logits = model(
                    src_input_ids = src_ids,
                    src_attn_mask = src_mask,
                    tgt_input_ids = tgt_ids,
                    tgt_lang_ids  = lang_ids,
                    tgt_attn_mask = tgt_mask,
                )
                loss = criterion(logits, labels) / CFG.GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (global_step + 1) % CFG.GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            train_loss = loss.item() * CFG.GRAD_ACCUM_STEPS
            train_ppl  = math.exp(min(train_loss, 20.0))

            if global_step % 100 == 0:
                elapsed = time.time() - t_start
                lr_enc  = optimizer.param_groups[0]["lr"]
                lr_dec  = optimizer.param_groups[1]["lr"]
                log.info(
                    f"step={global_step:>7d}  loss={train_loss:.4f}  ppl={train_ppl:.2f}  "
                    f"lr_enc={lr_enc:.2e}  lr_dec={lr_dec:.2e}  elapsed={elapsed:.0f}s"
                )
                log_metrics(
                    {
                        "step":       global_step,
                        "train_loss": train_loss,
                        "train_ppl":  train_ppl,
                        "lr_enc":     lr_enc,
                        "lr_dec":     lr_dec,
                        "elapsed_s":  round(elapsed, 2),
                    },
                    CFG.LOG_FILE,
                )

            # ── Evaluation ────────────────────────────────────────────────────
            if global_step > 0 and global_step % CFG.EVAL_EVERY == 0:
                model.eval()
                val_loss_total = 0.0
                val_batches    = 0
                with torch.no_grad():
                    for val_batch in val_loader:
                        v_src_ids  = val_batch["src_input_ids"].to(device)
                        v_src_mask = val_batch["src_attn_mask"].to(device)
                        v_tgt_ids  = val_batch["tgt_input_ids"].to(device)
                        v_tgt_mask = val_batch["tgt_attn_mask"].to(device)
                        v_labels   = val_batch["labels"].to(device)
                        v_lang_ids = val_batch["tgt_lang_id"].to(device)

                        with autocast(enabled=(device.type == "cuda")):
                            v_logits = model(
                                src_input_ids = v_src_ids,
                                src_attn_mask = v_src_mask,
                                tgt_input_ids = v_tgt_ids,
                                tgt_lang_ids  = v_lang_ids,
                                tgt_attn_mask = v_tgt_mask,
                            )
                            v_loss = criterion(v_logits, v_labels)
                        val_loss_total += v_loss.item()
                        val_batches    += 1
                        if val_batches >= 200:
                            break

                val_loss = val_loss_total / max(val_batches, 1)
                val_ppl  = math.exp(min(val_loss, 20.0))

                # BLEU
                base_model = model.module if hasattr(model, "module") else model
                bleu_score = evaluate_bleu(base_model, val_loader, tokenizer, device)

                log.info(
                    f"[EVAL step={global_step}]  val_loss={val_loss:.4f}  "
                    f"val_ppl={val_ppl:.2f}  BLEU={bleu_score:.2f}"
                )
                log_metrics(
                    {
                        "step":      global_step,
                        "val_loss":  val_loss,
                        "val_ppl":   val_ppl,
                        "bleu":      bleu_score,
                        "type":      "eval",
                    },
                    CFG.LOG_FILE,
                )

                # Early stopping
                if val_loss < best_val_loss:
                    best_val_loss    = val_loss
                    no_improve_count = 0
                else:
                    no_improve_count += 1
                    log.info(f"No improvement ({no_improve_count}/{CFG.EARLY_STOP_PATIENCE})")
                    if no_improve_count >= CFG.EARLY_STOP_PATIENCE:
                        log.info("Early stopping triggered.")
                        return

                model.train()

            # ── Checkpoint ───────────────────────────────────────────────────
            if global_step > 0 and global_step % CFG.SAVE_EVERY == 0:
                base_model = model.module if hasattr(model, "module") else model
                ckpt = save_checkpoint(
                    base_model, optimizer, scheduler, scaler,
                    global_step, best_val_loss, CFG.CHECKPOINT_DIR
                )
                if global_step % CFG.HUB_PUSH_EVERY == 0:
                    push_checkpoint_to_hub(ckpt, CFG.HF_REPO_ID, CFG.HF_TOKEN)

            global_step += 1

    log.info(f"Training complete. Best val loss: {best_val_loss:.4f}")


if __name__ == "__main__":
    train()
"""

dest = pathlib.Path("/kaggle/working/train_hybrid_model.py")
dest.parent.mkdir(parents=True, exist_ok=True)
dest.write_text(_SRC, encoding="utf-8")
print(f"✅ Written {len(_SRC)} chars → {dest}")


In [ ]:
# Verify both files exist and are readable
import pathlib
for fname in ["dataset_builder.py", "train_hybrid_model.py"]:
    p = pathlib.Path(f"/kaggle/working/{fname}")
    if p.exists():
        lines = len(p.read_text(encoding="utf-8").splitlines())
        print(f"✅  {fname:35s} {lines} lines  ({p.stat().st_size/1024:.1f} KB)")
    else:
        print(f"❌  {fname} NOT FOUND — something went wrong above!")

## Step 5 — Build the Multilingual Dataset

In [ ]:
import subprocess, sys, time, os

print("=" * 60)
print("RUNNING: dataset_builder.py")
print("=" * 60)
print("This downloads ~15-50 GB of corpora from HuggingFace.")
print("Expected time: 30-90 min depending on network speed.\n")

t0 = time.time()
result = subprocess.run(
    [sys.executable, "/kaggle/working/dataset_builder.py"],
    env=os.environ.copy(),
    text=True,
    capture_output=False,   # stream output directly to notebook
)
elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"⏱️  Completed in {elapsed/60:.1f} min  (exit code {result.returncode})")
if result.returncode != 0:
    print("❌  dataset_builder.py FAILED — check output above.")
else:
    print("✅  Dataset build complete!")

## Step 5b — Dataset Statistics

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

stats_path = pathlib.Path("/kaggle/working/multilingual_dataset/dataset_statistics.json")

if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)

    print("Dataset Statistics")
    def _n(v): return f"{v:,}" if isinstance(v, int) else str(v)
    print(f"  Total examples : {_n(stats.get('total_examples', 'N/A'))}")
    print(f"  Train split    : {_n(stats.get('train_examples', 'N/A'))}")
    print(f"  Val split      : {_n(stats.get('val_examples',   'N/A'))}")
    print(f"  Test split     : {_n(stats.get('test_examples',  'N/A'))}")
    print(f"  Language pairs : {stats.get('language_pairs', 'N/A')}")

    pairs = stats.get("pair_counts", {})
    if pairs:
        df_pairs = pd.Series(pairs).sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(14, 5))
        df_pairs.plot(kind="bar", ax=ax, color="steelblue")
        ax.set_title("Rows per Language Pair", fontsize=13, fontweight="bold")
        ax.set_ylabel("Count"); ax.set_xlabel("Language Pair")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

    domains = stats.get("domain_distribution", {})
    if domains:
        df_dom = pd.Series(domains)
        fig2, ax2 = plt.subplots(figsize=(6, 4))
        df_dom.plot(kind="pie", ax=ax2, autopct="%1.1f%%")
        ax2.set_title("Domain Distribution"); ax2.set_ylabel("")
        plt.tight_layout(); plt.show()
else:
    print(f"⚠️  Statistics file not found at {stats_path}")
    print("    This is expected if dataset_builder.py hasn't finished yet.")
    # List what IS in the directory
    work_dir = pathlib.Path("/kaggle/working/multilingual_dataset")
    if work_dir.exists():
        files = list(work_dir.iterdir())
        print(f"\n  Files in {work_dir}: {[f.name for f in files[:20]]}")
    else:
        print(f"\n  Directory {work_dir} does not exist yet.")

## Step 6 — Train the Hybrid Model

In [ ]:
import subprocess, sys, time, os

print("=" * 60)
print("RUNNING: train_hybrid_model.py")
print("=" * 60)
print("Training custom-hybrid-v1 (XLM-RoBERTa + 6-layer decoder).")
print("Expected time: 4-8 hours on 2×T4 GPUs.\n")

t0 = time.time()
result = subprocess.run(
    [sys.executable, "/kaggle/working/train_hybrid_model.py"],
    env=os.environ.copy(),
    text=True,
    capture_output=False,
)
elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"⏱️  Training finished in {elapsed/3600:.2f} hours  (exit code {result.returncode})")
if result.returncode != 0:
    print("❌  Training FAILED — check output above.")
else:
    print("✅  Training complete! Check Hub for pushed checkpoints.")

## Step 7 — Training Metrics Visualisation

In [ ]:
import json, pathlib
import pandas as pd
import matplotlib.pyplot as plt

log_path = pathlib.Path("/kaggle/working/checkpoints/training_log.jsonl")

if log_path.exists():
    records = []
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    df_log = pd.DataFrame(records)
    print(f"Total log entries: {len(df_log)}")
    print(df_log.tail(5).to_string(index=False))

    train_df = df_log[df_log.get("type", pd.NA) != "eval"] if "type" in df_log.columns else df_log
    eval_df  = df_log[df_log.get("type", "") == "eval"] if "type" in df_log.columns else pd.DataFrame()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor("#0d0d14")
    for ax in axes:
        ax.set_facecolor("#1a1a2e")
        ax.tick_params(colors="#9898b8"); ax.yaxis.label.set_color("#9898b8")
        ax.xaxis.label.set_color("#9898b8"); ax.title.set_color("#f0f0ff")

    if not train_df.empty and "train_loss" in train_df.columns:
        axes[0].plot(train_df["step"], train_df["train_loss"], color="#6c63ff", lw=1.5)
        axes[0].set_title("Training Loss"); axes[0].set_xlabel("Step"); axes[0].set_ylabel("Loss")
        axes[0].grid(alpha=0.2)

    if not eval_df.empty and "val_loss" in eval_df.columns:
        axes[1].plot(eval_df["step"], eval_df["val_loss"], color="#ef4444", marker="o", lw=1.5, ms=4)
        axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Step"); axes[1].set_ylabel("Val Loss")
        axes[1].grid(alpha=0.2)

    if not eval_df.empty and "bleu" in eval_df.columns:
        axes[2].plot(eval_df["step"], eval_df["bleu"], color="#10b981", marker="s", lw=1.5, ms=4)
        axes[2].set_title("BLEU Score (Val)"); axes[2].set_xlabel("Step"); axes[2].set_ylabel("BLEU")
        axes[2].grid(alpha=0.2)

    plt.suptitle("custom-hybrid-v1 Training Metrics", fontsize=14, fontweight="bold", color="#f0f0ff")
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_metrics.png", dpi=150, facecolor="#0d0d14")
    plt.show()
    print("\n📊 Plot saved → /kaggle/working/training_metrics.png")
else:
    print(f"⚠️  Log file not found at {log_path}")
    print("    This is expected if training hasn't run yet.")
    checkpoints = list(pathlib.Path("/kaggle/working/checkpoints").glob("*")) \
        if pathlib.Path("/kaggle/working/checkpoints").exists() else []
    print(f"    Checkpoint dir contents: {[p.name for p in checkpoints[:10]]}")

## Step 8 — List Saved Checkpoints & Hub Links

In [ ]:
import pathlib, os

ckpt_dir = pathlib.Path("/kaggle/working/checkpoints")
HF_USERNAME = os.environ.get("HF_USERNAME", "")

if ckpt_dir.exists():
    checkpoints = sorted(ckpt_dir.glob("checkpoint_step_*.pt"))
    if checkpoints:
        print(f"Found {len(checkpoints)} checkpoint(s):\n")
        for c in checkpoints:
            size_mb = c.stat().st_size / 1e6
            print(f"  ✅  {c.name:<40s}  {size_mb:.1f} MB")
    else:
        # Look for any checkpoint files
        all_files = list(ckpt_dir.iterdir())
        print(f"No checkpoint_step_*.pt files. Contents of {ckpt_dir}:")
        for f in sorted(all_files)[:20]:
            print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)" if f.is_file() else f"  {f.name}/")
else:
    print(f"⚠️  Checkpoint directory not found: {ckpt_dir}")
    print("    Training may not have run yet.")

print()
print("=" * 55)
print("📋 Phase 1 Deliverables Summary")
print("=" * 55)
print(f"  Dataset (Parquet)  : /kaggle/working/multilingual_dataset/")
print(f"  Statistics JSON    : /kaggle/working/multilingual_dataset/dataset_statistics.json")
print(f"  Checkpoints        : /kaggle/working/checkpoints/")
print(f"  Training log       : /kaggle/working/checkpoints/training_log.jsonl")
print(f"  Metrics plot       : /kaggle/working/training_metrics.png")
if HF_USERNAME:
    print(f"\n  HF Dataset repo    : https://huggingface.co/datasets/{HF_USERNAME}/multilingual-translation-data")
    print(f"  HF Model repo      : https://huggingface.co/{HF_USERNAME}/custom-hybrid-v1")
else:
    print("\n  ⚠️  Set HF_USERNAME to see Hub repo links")
print("\n✅  Phase 1 complete — proceed to Phase 2 notebook!")

---
## ✅ Phase 1 Complete

**Next:** Upload `phase2_notebook.ipynb` to a new Kaggle notebook for NLLB QLoRA + Whisper LoRA fine-tuning.